# Monte Carlo AI Inference: run the experiments on Google Colab

This notebook builds the Monte Carlo llama.cpp from this repository, downloads the pinned model and
dataset (both sha256-verified), and runs the ablation sweep. Results are written to your Google
Drive, so a disconnected session loses nothing: run the notebook again and the sweep resumes
where it stopped.

**Runtime.** The Monte Carlo runs use the CPU reference path, so a GPU does not help. Speed scales with
the number of CPU cores; the next cell prints how many you have. Free Colab gives 2 vCPUs, which is
enough for the smoke test (a few minutes) but slow for the full 8B sweep (42 settings), roughly an
hour per setting at 100 chunks, compared with about 9 minutes on 16 vCPUs. To finish the full
sweep on Colab, use a runtime with more CPUs, or split it with `SHARD` over several sessions
(see REPRODUCE.md, "Splitting the sweep").

In [ ]:
!nproc && free -g | head -2 && lscpu | grep -E 'Model name'

## 1. Settings

- `MODEL`: `qwen2.5-0.5b-q4km` (smoke test, small) or `llama3.1-8b-q4km` / `qwen2.5-7b-q4km` (paper models).
- `CHUNKS`: WikiText-2 chunks of 512 tokens per run. The paper uses 100; use 5 for a quick check.
- `SHARD`: `"1/1"` runs everything. `"k/n"` runs every n-th setting starting at k, so n sessions
  (or n people) can share one sweep.
- `RESULTS_DIR`: where run folders go. Google Drive keeps them across sessions.

In [ ]:
import os
MODEL       = "qwen2.5-0.5b-q4km"
CHUNKS      = "5"
SHARD       = "1/1"
USE_DRIVE   = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_DIR = "/content/drive/MyDrive/mc-inference-results"
else:
    RESULTS_DIR = "/content/mc-inference-results"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ.update(MODEL=MODEL, CHUNKS=CHUNKS, SHARD=SHARD, RESULTS_DIR=RESULTS_DIR, RESUME="1",
                  WORK_DIR="/content/mc-work")
print("results go to", RESULTS_DIR)

## 2. Get the code and build (about 10 minutes on 2 vCPUs)

In [ ]:
%%bash
set -e
cd /content
[ -d Monte-Carlo-AI-Inference ] || git clone --depth 1 https://github.com/syncaissa/Monte-Carlo-AI-Inference.git
cd Monte-Carlo-AI-Inference
scripts/setup.sh
scripts/build.sh 2>&1 | tail -5

## 3. Download the model and dataset (sha256-verified)

In [ ]:
%%bash
set -e
cd /content/Monte-Carlo-AI-Inference
scripts/get_model.sh "$MODEL"
scripts/get_data.sh

## 4. Check that the build is exact

`exact` mode routes every matmul through the Monte Carlo code with a 100% budget; its perplexity
must equal `off` (the unmodified llama.cpp path) to every printed digit.

In [ ]:
%%bash
cd /content/Monte-Carlo-AI-Inference
BUDGETS="" SHARD=1/1 scripts/sweep_mc.sh "$MODEL" 2>&1 | grep -vE '^\[' | tail -4

## 5. Run the sweep

Each setting writes one folder to `RESULTS_DIR` and prints its summary line. Settings that already
have a result there (with the same `CHUNKS`) are skipped, so if the session disconnects, run
cells 1 to 3 again and then this one.

In [ ]:
%%bash
cd /content/Monte-Carlo-AI-Inference
LIST=1 scripts/sweep_mc.sh "$MODEL" 2>&1 | grep -c '/' | xargs echo "settings left in this shard:"
scripts/sweep_mc.sh "$MODEL" 2>&1 | grep -E '^\[.*setting|^[a-z]' | grep -v '^model'

## 6. Results

One row per run. `read_frac` is the fraction of weight bytes read, `rel_err` the relative matmul
error. To share your numbers, zip the folder (next cell) and attach it to a GitHub issue or pull
request.

In [ ]:
%%bash
cd /content/Monte-Carlo-AI-Inference
scripts/collect_results.sh "$MODEL"
cd "$RESULTS_DIR"/.. && zip -qr /content/mc-inference-results.zip "$(basename "$RESULTS_DIR")" \
    && echo "zipped to /content/mc-inference-results.zip"

In [ ]:
from google.colab import files
files.download("/content/mc-inference-results.zip")